# Fairness Research — Colab + Google Drive

**Golden rule:** Every result row is saved to Google Drive *before* the next row runs.

**When Colab disconnects:** Run **cells 1 → 2** again, then jump to whatever phase cell you need (4–10). Completed rows are skipped automatically.

Prerequisites: folder on Drive at `MyDrive/fairness_research/`, HF token in Colab Secrets as `HF_TOKEN`.

In [1]:
# CELL 1 — Run at the start of EVERY new Colab session
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/fairness_research'

!pip install -q transformers accelerate bitsandbytes sentence-transformers \
    pandas numpy scipy statsmodels seaborn matplotlib scikit-learn tqdm huggingface_hub

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00


In [2]:
# CELL 2 — Session bootstrap + HF login + resume status (run every new session)
import os, sys
from huggingface_hub import login

DRIVE_ROOT = '/content/drive/MyDrive/fairness_research'
os.environ['FAIRNESS_DRIVE_ROOT'] = DRIVE_ROOT

# HF token from Colab Secrets (key icon) OR paste when prompted
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN') or input('Paste HF token: ').strip()

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN)
print('HF login OK')

# Idempotent setup — does NOT reset checkpoints or progress.json
!python {DRIVE_ROOT}/scripts/setup_drive.py

sys.path.insert(0, f'{DRIVE_ROOT}/code')
os.chdir(f'{DRIVE_ROOT}/code')

from src.colab_session import print_resume_status
from pathlib import Path
print_resume_status(Path(DRIVE_ROOT))

Paste HF token: REDACTED


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login OK
Kept existing experiment_config.json (same version)
Kept existing prompts_v1.csv
Kept existing progress.json (resume-safe)
Synced code/ from uploaded project folder
Setup OK at /content/drive/MyDrive/fairness_research

=== RESUME STATUS (from Google Drive) ===
Phase: judge_secondary
  generations: 3892 rows
  scores_primary: 3886 rows
  scores_secondary: 372 rows
  embeddings: 0 rows
  failures: 14 rows
Last run_key: meta-llama/Llama-3.1-8B-Instruct|occupation_3_c_explicit|1.5|1.0|456
Manifest: /content/drive/MyDrive/fairness_research/state/MANIFEST.json
Summary:  /content/drive/MyDrive/fairness_research/state/SESSION_SUMMARY.txt
Export:   /content/drive/MyDrive/fairness_research/exports/fairness_experiment_latest.zip



In [ ]:
# CELL 3 — Pilot generation (~8 rows, resume-safe)
import os
os.chdir('/content/drive/MyDrive/fairness_research/code')
os.environ['FAIRNESS_DRIVE_ROOT'] = '/content/drive/MyDrive/fairness_research'
!python run_generation.py --pilot

Drive root: /content/drive/MyDrive/fairness_research
Skipping 0 completed rows. 8 remaining.

=== Loading meta-llama/Llama-3.1-8B-Instruct ===
Llama-3.1-8B-Instruct:   0% 0/8 [00:00<?, ?it/s]
config.json: 100% 855/855 [00:00<00:00, 4.74MB/s]

tokenizer_config.json: 100% 55.4k/55.4k [00:00<00:00, 80.8MB/s]

tokenizer.json:   0% 0.00/9.09M [00:00<?, ?B/s]
tokenizer.json: 100% 9.09M/9.09M [00:00<00:00, 73.3MB/s]

special_tokens_map.json: 100% 296/296 [00:00<00:00, 1.25MB/s]

model.safetensors.index.json: 100% 23.9k/23.9k [00:00<00:00, 62.3MB/s]


Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]

Fetching 4 files: 100% 4/4 [04:45<00:00, 71.30s/it] 

Download complete: 100% 16.1G/16.1G [04:45<00:00, 56.2MB/s]

Loading weights:   0% 0/291 [00:00<?, ?it/s]
Loading weights:   0% 1/291 [00:04<23:45,  4.92s/it]
Loading weights:   1% 2/291 [00:09<22:08,  4.60s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyT

In [ ]:
# CELL 4 — Pilot judging (resume-safe) — auto-fixes broken files on Drive
import os, shutil

DRIVE = '/content/drive/MyDrive/fairness_research'
CODE = f'{DRIVE}/code'
os.makedirs(f'{CODE}/src', exist_ok=True)

# Fix 1: run_judge.py missing "from pathlib import Path"
rj = f'{CODE}/run_judge.py'
text = open(rj).read()
if 'from pathlib import Path' not in text:
    text = text.replace(
        'import argparse\nimport os\n\nimport pandas',
        'import argparse\nimport os\nimport sys\nfrom pathlib import Path\n\nimport pandas',
    )
    open(rj, 'w').write(text)
    print('Patched run_judge.py')
else:
    print('run_judge.py OK')

# Fix 2: judge_hf.py broken template
src_mac = f'{DRIVE}/src/judge_hf.py'
dst = f'{CODE}/src/judge_hf.py'
if os.path.exists(src_mac):
    shutil.copy2(src_mac, dst)
    print('Copied judge_hf.py from Drive src/')
elif os.path.exists(dst) and 'build_judge_prompt' not in open(dst).read():
    t = open(dst).read()
    t = t.replace('{ text }', '__GENERATED_TEXT__').replace('{text}', '__GENERATED_TEXT__')
    if 'def build_judge_prompt' not in t:
        t = t.replace(
            'def parse_judge_response',
            'def build_judge_prompt(generated_text: str) -> str:\n'
            '    return JUDGE_TEMPLATE.replace("__GENERATED_TEXT__", generated_text[:3000])\n\n\n'
            'def parse_judge_response',
        )
    t = t.replace('JUDGE_TEMPLATE.format(text=generated_text[:3000])', 'build_judge_prompt(generated_text)')
    open(dst, 'w').write(t)
    print('Patched judge_hf.py in code/src/')
else:
    print('judge_hf.py OK')

os.chdir(CODE)
os.environ['FAIRNESS_DRIVE_ROOT'] = DRIVE
!python run_judge.py --phase primary

run_judge.py OK
Copied judge_hf.py from Drive src/
Judge: meta-llama/Llama-3.1-8B-Instruct | Phase: primary | 4 rows to score
Loading weights:   1% 2/291 [00:08<21:36,  4.49s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 291/291 [01:08<00:00,  4.26it/s]
  0% 0/4 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctua

In [ ]:
# CELL 5 — RESUME TEST: re-run pilot generation — should say "0 remaining" / skip all
import os
os.chdir('/content/drive/MyDrive/fairness_research/code')
os.environ['FAIRNESS_DRIVE_ROOT'] = '/content/drive/MyDrive/fairness_research'
!python run_generation.py --pilot

Drive root: /content/drive/MyDrive/fairness_research
Skipping 8 completed rows. 4 remaining.

=== Loading meta-llama/Llama-3.1-8B-Instruct ===
Llama-3.1-8B-Instruct:   0% 0/4 [00:00<?, ?it/s]
Loading weights:   0% 0/291 [00:00<?, ?it/s]
Loading weights:   0% 1/291 [00:04<21:49,  4.52s/it]
Loading weights:   1% 2/291 [00:08<20:50,  4.33s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)

Loading weights:   1% 4/291 [00:09<08:39,  1.81s/it]
Loading weights:   2% 5/291 [00:09<06:40,  1.40s/it]
Loading weights:   2% 6/291 [00:10<05:18,  1.12s/it]
Loading weights:   3% 9/291 [00:10<02:25,  1.93it/s]
Loading weights:   3% 10/291 [00:10<02:02,  2.30it/s]
Loading weights:   4% 13/291 [00:11<01:25,  3.25it/s]
Loading weights:   5% 14/291 [00:11<01:33,  2.97it/s]
Loading weights:   5% 15/291 [0

## Full experiment (across many sessions — re-run Cell 1–2 then the phase you need)

In [ ]:
# CELL 6 — Full generation (resume-safe; re-run until "0 remaining")
import os, json
DRIVE_ROOT = '/content/drive/MyDrive/fairness_research'
cfg_path = f'{DRIVE_ROOT}/experiment_config.json'
with open(cfg_path) as f:
    cfg = json.load(f)
cfg['pilot']['enabled'] = False  # saved to Drive — stays off on resume
with open(cfg_path, 'w') as f:
    json.dump(cfg, f, indent=2)

os.chdir(f'{DRIVE_ROOT}/code')
os.environ['FAIRNESS_DRIVE_ROOT'] = DRIVE_ROOT
!python run_generation.py

Drive root: /content/drive/MyDrive/fairness_research
Skipping 3799 completed rows. 93 remaining.

=== Loading mistralai/Mistral-7B-Instruct-v0.3 ===
Mistral-7B-Instruct-v0.3:   0% 0/93 [00:00<?, ?it/s]
config.json: 100% 601/601 [00:00<00:00, 2.94MB/s]

tokenizer_config.json: 100% 141k/141k [00:00<00:00, 108MB/s]

tokenizer.json: 100% 1.96M/1.96M [00:00<00:00, 74.5MB/s]

tokenizer.model:   0% 0.00/587k [00:00<?, ?B/s]
tokenizer.model:   0% 0.00/587k [00:00<?, ?B/s]
tokenizer.model:   0% 0.00/587k [00:00<?, ?B/s]
tokenizer.model:   0% 0.00/587k [00:00<?, ?B/s]
tokenizer.model: 100% 587k/587k [00:00<00:00, 762kB/s]

special_tokens_map.json: 100% 414/414 [00:00<00:00, 2.25MB/s]

model.safetensors.index.json: 100% 23.9k/23.9k [00:00<00:00, 81.8MB/s]


Fetching 3 files:   0% 0/3 [00:00<?, ?it/s]

Fetching 3 files: 100% 3/3 [08:24<00:00, 168.22s/it]

Download complete: 100% 14.5G/14.5G [08:24<00:00, 28.7MB/s]

Loading weights:   0% 0/291 [00:00<?, ?it/s]
Loading weights:   0% 1/291 [00:01<06:

In [3]:
# CELL 7 — Primary judging (resume-safe) — auto-fixes judge files on Drive
import os, shutil

DRIVE = '/content/drive/MyDrive/fairness_research'
CODE = f'{DRIVE}/code'
os.makedirs(f'{CODE}/src', exist_ok=True)

# Fix 1: run_judge.py missing Path import
rj = f'{CODE}/run_judge.py'
text = open(rj).read()
if 'from pathlib import Path' not in text:
    text = text.replace(
        'import argparse\nimport os\n\nimport pandas',
        'import argparse\nimport os\nimport sys\nfrom pathlib import Path\n\nimport pandas',
    )
    open(rj, 'w').write(text)
    print('Patched run_judge.py')
else:
    print('run_judge.py OK')

# Fix 2: judge_hf.py broken template
src_mac = f'{DRIVE}/src/judge_hf.py'
dst = f'{CODE}/src/judge_hf.py'
if os.path.exists(src_mac):
    shutil.copy2(src_mac, dst)
    print('Copied judge_hf.py from Drive src/')
elif os.path.exists(dst) and 'build_judge_prompt' not in open(dst).read():
    t = open(dst).read()
    if 'def build_judge_prompt' not in t:
        t = t.replace(
            'def parse_judge_response',
            'def build_judge_prompt(generated_text: str) -> str:\n'
            '    return JUDGE_TEMPLATE.replace("__GENERATED_TEXT__", generated_text[:3000])\n\n\n'
            'def parse_judge_response',
        )
    t = t.replace('JUDGE_TEMPLATE.format(text=generated_text[:3000])', 'build_judge_prompt(generated_text)')
    open(dst, 'w').write(t)
    print('Patched judge_hf.py')
else:
    print('judge_hf.py OK')

os.chdir(CODE)
os.environ['FAIRNESS_DRIVE_ROOT'] = DRIVE
!python run_judge.py --phase primary

Patched run_judge.py
Copied judge_hf.py from Drive src/
Judge: meta-llama/Llama-3.1-8B-Instruct | Phase: primary | 1111 rows to score
config.json: 100% 855/855 [00:00<00:00, 4.56MB/s]
tokenizer_config.json: 100% 55.4k/55.4k [00:00<00:00, 38.4MB/s]
tokenizer.json: 100% 9.09M/9.09M [00:00<00:00, 22.2MB/s]
special_tokens_map.json: 100% 296/296 [00:00<00:00, 1.97MB/s]
model.safetensors.index.json: 100% 23.9k/23.9k [00:00<00:00, 63.5MB/s]
Fetching 4 files: 100% 4/4 [02:33<00:00, 38.32s/it] 
Download complete: 100% 16.1G/16.1G [02:33<00:00, 105MB/s]
Loading weights:   1% 2/291 [00:08<20:47,  4.32s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 291/291 [01:05<00:00,  4.44it/s]
generation_config.json: 100% 184/184 [00:00<00:00, 751kB/s]
  0% 0/1111 [00:00<?, ?it/s]/us

In [7]:
# CELL 8 (OVERWRITE) — write correct run_judge.py then run secondary (resume-safe)
import os

DRIVE_ROOT = '/content/drive/MyDrive/fairness_research'
os.chdir(f'{DRIVE_ROOT}/code')
os.environ['FAIRNESS_DRIVE_ROOT'] = DRIVE_ROOT

CORRECT = r'''"""Phase 2 & 3: Primary and secondary LLM-as-judge scoring."""

from __future__ import annotations

import argparse
import os
import sys
from pathlib import Path

import pandas as pd
from tqdm import tqdm

ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT))

from src.checkpoint import CheckpointManager, utc_now
from src.embed_stereoset import select_secondary_sample
from src.judge_hf import HFJudge
from src.paths import drive_root, load_config


def main(phase: str = "primary") -> None:
    config = load_config()
    drive = drive_root()
    os.environ.setdefault("FAIRNESS_DRIVE_ROOT", str(drive))
    ckpt = CheckpointManager(drive)

    gen_df = ckpt.load_csv("generations")
    if gen_df.empty:
        print("No generations found. Run generation phase first.")
        return

    if phase == "primary":
        csv_name = "scores_primary"
        judge_model = config["models"]["judge_primary"]
        skip = ckpt.load_run_keys(csv_name)
        todo = gen_df[~gen_df["run_key"].isin(skip)]
    else:
        csv_name = "scores_secondary"
        judge_model = config["models"]["judge_secondary"]
        skip = ckpt.load_run_keys(csv_name)
        primary_done = ckpt.load_run_keys("scores_primary")
        pool = gen_df[gen_df["run_key"].isin(primary_done)]
        sample_keys = select_secondary_sample(
            pool["run_key"],
            config.get("secondary_judge_fraction", 0.1),
            config.get("secondary_judge_sample_seed", 2026),
        )
        todo = pool[pool["run_key"].isin(sample_keys) & ~pool["run_key"].isin(skip)]

    print(f"Judge: {judge_model} | Phase: {phase} | {len(todo)} rows to score")

    judge = HFJudge(judge_model, load_in_4bit=config["generation"].get("load_in_4bit", True))
    skip_count = len(skip)
    saved = 0
    failed = 0
    try:
        judge.load()
        for _, row in tqdm(todo.iterrows(), total=len(todo)):
            try:
                result = judge.score(str(row["generated_text"]))
                if not result.get("parse_ok"):
                    failed += 1
                    ckpt.log_failure(
                        phase, row["run_key"],
                        model_name=row["model_name"],
                        prompt_id=row["prompt_id"],
                        temperature=row["temperature"],
                        top_p=row["top_p"],
                        seed=int(row["seed"]),
                        error_type="parse_error",
                        error_message=result.get("justification", "")[:200],
                    )
                    continue
                out = {
                    "run_key": row["run_key"],
                    "model_name": row["model_name"],
                    "prompt_id": row["prompt_id"],
                    "prompt_tier": row["prompt_tier"],
                    "demographic_dimension": row["demographic_dimension"],
                    "temperature": row["temperature"],
                    "top_p": row["top_p"],
                    "seed": int(row["seed"]),
                    "stereotype_score": result["stereotype_score"],
                    "stereotype_type": result["stereotype_type"],
                    "justification": result["justification"],
                    "judge_model": judge_model,
                    "timestamp": utc_now(),
                }
                ckpt.append_row(csv_name, out)
                saved += 1
                key = "scores_primary_done" if phase == "primary" else "scores_secondary_done"
                ckpt.save_progress(
                    phase=f"judge_{phase}",
                    **{key: skip_count + saved},
                    last_run_key=row["run_key"],
                )
            except Exception as e:
                failed += 1
                ckpt.log_failure(
                    phase, row["run_key"],
                    model_name=row["model_name"],
                    prompt_id=row["prompt_id"],
                    error_type=type(e).__name__,
                    error_message=str(e)[:500],
                )
    finally:
        judge.unload()

    ckpt.log_session(f"Judge {phase} done. saved={skip_count + saved} new={saved} failed={failed}")
    ckpt.sync_progress_counts()
    _export(drive)
    print(f"Judge {phase} complete. {saved} new rows saved ({skip_count + saved} total on Drive). {failed} failed this run.")


def _export(drive):
    try:
        from run_export import export_bundle
        z = export_bundle(drive)
        print(f"Saved export bundle: {z}")
    except Exception as e:
        print(f"Export skipped: {e}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--phase", choices=["primary", "secondary"], default="primary")
    args = parser.parse_args()
    main(phase=args.phase)
'''

with open('run_judge.py', 'w') as f:
    f.write(CORRECT)
print("Wrote correct run_judge.py")

!python run_judge.py --phase secondary

Wrote correct run_judge.py
Judge: mistralai/Mistral-7B-Instruct-v0.3 | Phase: secondary | 16 rows to score
config.json: 100% 601/601 [00:00<00:00, 3.47MB/s]
tokenizer_config.json: 100% 141k/141k [00:00<00:00, 218MB/s]
tokenizer.json: 100% 1.96M/1.96M [00:00<00:00, 79.4MB/s]
tokenizer.model: 100% 587k/587k [00:00<00:00, 925kB/s]
special_tokens_map.json: 100% 414/414 [00:00<00:00, 2.50MB/s]
model.safetensors.index.json: 100% 23.9k/23.9k [00:00<00:00, 55.2MB/s]
Fetching 3 files: 100% 3/3 [02:53<00:00, 57.77s/it]
Download complete: 100% 14.5G/14.5G [02:53<00:00, 83.6MB/s]
Loading weights:   1% 2/291 [00:02<05:41,  1.18s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 291/291 [00:59<00:00,  4.93it/s]
generation_config.json: 100% 116/116 [00:00<00:00, 719kB/s]
  0% 0

In [8]:
# CELL 9 — Embeddings (resume-safe, CPU OK)
import os
DRIVE_ROOT = '/content/drive/MyDrive/fairness_research'
os.chdir(f'{DRIVE_ROOT}/code')
os.environ['FAIRNESS_DRIVE_ROOT'] = DRIVE_ROOT
!python run_embed.py

Embeddings: 3838 rows remaining
Loading weights: 100% 199/199 [00:00<00:00, 795.95it/s]
100% 3838/3838 [13:44<00:00,  4.66it/s]
Saved export bundle: /content/drive/MyDrive/fairness_research/exports/fairness_experiment_latest.zip
Embeddings complete. 3892 rows saved.


In [9]:
# CELL 10 — Analysis + figures
import os
DRIVE_ROOT = '/content/drive/MyDrive/fairness_research'
os.chdir(f'{DRIVE_ROOT}/code')
os.environ['FAIRNESS_DRIVE_ROOT'] = DRIVE_ROOT
!python run_analyze.py

Running analysis...
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2054: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2245: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)
/usr/lo

## Export for download / share with assistant (run anytime)

In [10]:
# CELL 12 — Export zip to Drive + download to your computer
import os
from pathlib import Path
from google.colab import files

DRIVE_ROOT = '/content/drive/MyDrive/fairness_research'
os.chdir(f'{DRIVE_ROOT}/code')
os.environ['FAIRNESS_DRIVE_ROOT'] = DRIVE_ROOT

!python run_export.py

ZIP = f'{DRIVE_ROOT}/exports/fairness_experiment_latest.zip'
print('Zip on Drive:', ZIP)
print('Also read: state/SESSION_SUMMARY.txt and state/MANIFEST.json')

# Download to your Mac (upload this zip back to assistant in chat)
files.download(ZIP)

Export complete: /content/drive/MyDrive/fairness_research/exports/fairness_experiment_latest.zip
Size: 7.44 MB

Download this file and upload to your assistant.
Path on Drive: /content/drive/MyDrive/fairness_research/exports/fairness_experiment_latest.zip
Zip on Drive: /content/drive/MyDrive/fairness_research/exports/fairness_experiment_latest.zip
Also read: state/SESSION_SUMMARY.txt and state/MANIFEST.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
# CELL 11 — Progress check (anytime)
import os
from pathlib import Path
DRIVE_ROOT = '/content/drive/MyDrive/fairness_research'
os.environ['FAIRNESS_DRIVE_ROOT'] = DRIVE_ROOT
import sys
sys.path.insert(0, f'{DRIVE_ROOT}/code')
from src.colab_session import print_resume_status
print_resume_status(Path(DRIVE_ROOT))


=== RESUME STATUS (from Google Drive) ===
Phase: embed
  generations: 3892 rows
  scores_primary: 3886 rows
  scores_secondary: 388 rows
  embeddings: 3892 rows
  failures: 14 rows
Last run_key: mistralai/Mistral-7B-Instruct-v0.3|occupation_3_c_implicit|1.0|0.3|456
Manifest: /content/drive/MyDrive/fairness_research/state/MANIFEST.json
Summary:  /content/drive/MyDrive/fairness_research/state/SESSION_SUMMARY.txt
Export:   /content/drive/MyDrive/fairness_research/exports/fairness_experiment_latest.zip

